# Construcción de la dimensión de clientes — `silver.dim_cliente`

## Objetivo del notebook

Este notebook construye la primera tabla de la capa Silver: una versión limpia y enriquecida de la dimensión de clientes a partir de los datos crudos almacenados en `bronze.dim_cliente`.

La capa Silver representa el primer nivel de transformación analítica: los datos ya son comparables, los tipos están correctamente casteados, los identificadores son consistentes y se han añadido los campos derivados que el análisis posterior requerirá.

### Transformaciones que se aplicarán

A partir de los hallazgos de los notebooks de exploración (`01_exploracion_general` y `02_exploracion_mosaic`), las decisiones metodológicas a aplicar son:

1. Garantizar el tipo VARCHAR del `id_cliente` para coherencia con las tablas de hechos en JOINs posteriores.
2. Limpiar campos de texto (TRIM en nombre, dirección, localidad) para evitar problemas con espacios en blanco.
3. Normalizar el código postal a cinco dígitos mediante padding (`LPAD`), permitiendo el JOIN posterior con la tabla MOSAIC.
4. Crear el flag `es_cliente_espanol` aplicando una validación cruzada entre el código postal y el código de provincia (los dos primeros dígitos del CP español deben coincidir con el código de provincia).
5. Crear el campo `tipo_mercado` (`NACIONAL` o `INTERNACIONAL`) para diferenciar los dos grandes bloques de la cartera de Selmark.
6. Conservar el CP original en una columna paralela para auditoría y trazabilidad.

### Resultado esperado

Una tabla `silver.dim_cliente` con 3.469 filas (la misma cantidad que en bronze, sin pérdidas) y columnas limpias y enriquecidas, lista para ser utilizada como dimensión maestra de cliente en el resto del proyecto.

## 1. Configuración del entorno

Se establece la conexión con la base DuckDB en **modo escritura** (`read_only=False`). Es el primer notebook que requiere permisos de escritura, ya que va a crear una nueva tabla en el esquema `silver`.

In [30]:
import duckdb
import pandas as pd
from pathlib import Path

RUTA_PROYECTO = Path("..").resolve()
RUTA_DUCKDB = RUTA_PROYECTO / "duckdb" / "selmark.duckdb"

# Conexión en modo escritura
con = duckdb.connect(str(RUTA_DUCKDB), read_only=False)
print(f"Conexión establecida con: {RUTA_DUCKDB}")
print(f"Modo: escritura")

Conexión establecida con: C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\duckdb\selmark.duckdb
Modo: escritura


## 2. Inspección previa de la tabla origen

Antes de transformar, se confirma el estado actual de `bronze.dim_cliente`: número de filas, esquema y muestra de los datos. Esta verificación es importante por si la base se hubiera modificado entre sesiones.

In [31]:
# Conteo y esquema
print(f"Filas en bronze.dim_cliente: {con.execute('SELECT COUNT(*) FROM bronze.dim_cliente').fetchone()[0]:,}\n")

print("Esquema:")
esquema = con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'bronze' AND table_name = 'dim_cliente'
    ORDER BY ordinal_position
""").fetchdf()
print(esquema.to_string(index=False))

print("\nMuestra de 5 filas:")
muestra = con.execute("SELECT * FROM bronze.dim_cliente LIMIT 5").fetchdf()
print(muestra.to_string(index=False))

Filas en bronze.dim_cliente: 3,469

Esquema:
             column_name data_type
              id_cliente   VARCHAR
          nombre_cliente   VARCHAR
nombre_comercial_cliente   VARCHAR
       direccion_cliente   VARCHAR
       localidad_cliente   VARCHAR
   codigo_postal_cliente   VARCHAR
codigo_provincia_cliente   VARCHAR

Muestra de 5 filas:
id_cliente          nombre_cliente nombre_comercial_cliente                   direccion_cliente localidad_cliente codigo_postal_cliente codigo_provincia_cliente
         1 PEREZ RODRIGUEZ, AMADOR                      NaN      CL/ CELSO E. FERREIRO - 7 8º A              VIGO                 36203                      036
      1010    PERFECT FIT LINGERIE                      NaN                  16 COURT STREET S.       THONDER BAY                P7B2W3                      CAN
      1024        PAULA-COM S.R.L.        Litvinov Alexandr                ALBA IULIA, 2, AP.12          CHISINAU                MD2064                       MD
      1037

## 3. Construcción de `silver.dim_cliente`

Se ejecuta una sentencia `CREATE OR REPLACE TABLE` que aplica todas las transformaciones en una sola operación. Las claves del proceso son:

- `id_cliente` confirmado como VARCHAR mediante casteo explícito.
- TRIM sobre todos los campos de texto.
- Normalización del CP: si el campo tiene entre 1 y 5 caracteres y son todos dígitos, se aplica padding a 5. En caso contrario se conserva sin transformar.
- Flag `es_cliente_espanol`: validación cruzada estricta entre CP y código de provincia.
- Campo `tipo_mercado`: `NACIONAL` o `INTERNACIONAL` derivado del flag.
- Conservación del CP original en `codigo_postal_original`.

In [32]:
con.execute("""
    CREATE OR REPLACE TABLE silver.dim_cliente AS
    SELECT
        CAST(id_cliente AS VARCHAR) AS id_cliente,
        TRIM(nombre_cliente) AS nombre_cliente,
        TRIM(nombre_comercial_cliente) AS nombre_comercial_cliente,
        TRIM(direccion_cliente) AS direccion_cliente,
        TRIM(localidad_cliente) AS localidad_cliente,
        
        -- CP original para auditoría
        TRIM(codigo_postal_cliente) AS codigo_postal_original,
        
        -- CP normalizado: padding a 5 dígitos solo si es numérico válido
        CASE
            WHEN TRIM(codigo_postal_cliente) IS NULL THEN NULL
            WHEN LENGTH(TRIM(codigo_postal_cliente)) BETWEEN 1 AND 5
                 AND REGEXP_MATCHES(TRIM(codigo_postal_cliente), '^[0-9]+$')
            THEN LPAD(TRIM(codigo_postal_cliente), 5, '0')
            ELSE TRIM(codigo_postal_cliente)
        END AS codigo_postal_norm,
        
        -- Provincia: normalizar texto 'null' a NULL real
        CASE
            WHEN LOWER(TRIM(codigo_provincia_cliente)) IN ('null', 'nan', '') THEN NULL
            ELSE TRIM(codigo_provincia_cliente)
        END AS codigo_provincia_cliente,
        
        -- Flag de cliente español: 4 reglas combinadas con OR
        CASE
            -- REGLA A: provincia 2 dígitos válida que coincide con CP
            WHEN TRIM(codigo_postal_cliente) IS NOT NULL
                 AND LENGTH(TRIM(codigo_postal_cliente)) BETWEEN 1 AND 5
                 AND REGEXP_MATCHES(TRIM(codigo_postal_cliente), '^[0-9]+$')
                 AND TRIM(codigo_provincia_cliente) IS NOT NULL
                 AND LENGTH(TRIM(codigo_provincia_cliente)) = 2
                 AND REGEXP_MATCHES(TRIM(codigo_provincia_cliente), '^[0-9]{2}$')
                 AND TRY_CAST(TRIM(codigo_provincia_cliente) AS INTEGER) BETWEEN 1 AND 52
                 AND SUBSTR(LPAD(TRIM(codigo_postal_cliente), 5, '0'), 1, 2) = TRIM(codigo_provincia_cliente)
            THEN TRUE
            
            -- REGLA B: provincia '0XX' donde los 2 últimos coinciden con CP
            WHEN TRIM(codigo_postal_cliente) IS NOT NULL
                 AND LENGTH(TRIM(codigo_postal_cliente)) BETWEEN 1 AND 5
                 AND REGEXP_MATCHES(TRIM(codigo_postal_cliente), '^[0-9]+$')
                 AND TRIM(codigo_provincia_cliente) IS NOT NULL
                 AND LENGTH(TRIM(codigo_provincia_cliente)) = 3
                 AND REGEXP_MATCHES(TRIM(codigo_provincia_cliente), '^0[0-9]{2}$')
                 AND TRY_CAST(SUBSTR(TRIM(codigo_provincia_cliente), 2, 2) AS INTEGER) BETWEEN 1 AND 52
                 AND SUBSTR(LPAD(TRIM(codigo_postal_cliente), 5, '0'), 1, 2) = SUBSTR(TRIM(codigo_provincia_cliente), 2, 2)
            THEN TRUE
            
            -- REGLA C: provincia no informativa (NULL real o texto 'null') + CP español válido
            WHEN (
                    codigo_provincia_cliente IS NULL
                    OR LOWER(TRIM(codigo_provincia_cliente)) IN ('null', 'nan', '')
                 )
                 AND TRIM(codigo_postal_cliente) IS NOT NULL
                 AND LENGTH(TRIM(codigo_postal_cliente)) = 5
                 AND REGEXP_MATCHES(TRIM(codigo_postal_cliente), '^[0-9]+$')
                 AND TRY_CAST(SUBSTR(TRIM(codigo_postal_cliente), 1, 2) AS INTEGER) BETWEEN 1 AND 52
            THEN TRUE
            
            ELSE FALSE
        END AS es_cliente_espanol,
        
        -- Tipo de mercado: NACIONAL si cumple alguna regla, INTERNACIONAL en caso contrario
        CASE
            -- REGLA A
            WHEN TRIM(codigo_postal_cliente) IS NOT NULL
                 AND LENGTH(TRIM(codigo_postal_cliente)) BETWEEN 1 AND 5
                 AND REGEXP_MATCHES(TRIM(codigo_postal_cliente), '^[0-9]+$')
                 AND TRIM(codigo_provincia_cliente) IS NOT NULL
                 AND LENGTH(TRIM(codigo_provincia_cliente)) = 2
                 AND REGEXP_MATCHES(TRIM(codigo_provincia_cliente), '^[0-9]{2}$')
                 AND TRY_CAST(TRIM(codigo_provincia_cliente) AS INTEGER) BETWEEN 1 AND 52
                 AND SUBSTR(LPAD(TRIM(codigo_postal_cliente), 5, '0'), 1, 2) = TRIM(codigo_provincia_cliente)
            THEN 'NACIONAL'
            
            -- REGLA B
            WHEN TRIM(codigo_postal_cliente) IS NOT NULL
                 AND LENGTH(TRIM(codigo_postal_cliente)) BETWEEN 1 AND 5
                 AND REGEXP_MATCHES(TRIM(codigo_postal_cliente), '^[0-9]+$')
                 AND TRIM(codigo_provincia_cliente) IS NOT NULL
                 AND LENGTH(TRIM(codigo_provincia_cliente)) = 3
                 AND REGEXP_MATCHES(TRIM(codigo_provincia_cliente), '^0[0-9]{2}$')
                 AND TRY_CAST(SUBSTR(TRIM(codigo_provincia_cliente), 2, 2) AS INTEGER) BETWEEN 1 AND 52
                 AND SUBSTR(LPAD(TRIM(codigo_postal_cliente), 5, '0'), 1, 2) = SUBSTR(TRIM(codigo_provincia_cliente), 2, 2)
            THEN 'NACIONAL'
            
            -- REGLA C
            WHEN (
                    codigo_provincia_cliente IS NULL
                    OR LOWER(TRIM(codigo_provincia_cliente)) IN ('null', 'nan', '')
                 )
                 AND TRIM(codigo_postal_cliente) IS NOT NULL
                 AND LENGTH(TRIM(codigo_postal_cliente)) = 5
                 AND REGEXP_MATCHES(TRIM(codigo_postal_cliente), '^[0-9]+$')
                 AND TRY_CAST(SUBSTR(TRIM(codigo_postal_cliente), 1, 2) AS INTEGER) BETWEEN 1 AND 52
            THEN 'NACIONAL'
            
            ELSE 'INTERNACIONAL'
        END AS tipo_mercado
        
    FROM bronze.dim_cliente
""")

print("✅ Tabla silver.dim_cliente recreada con las 3 reglas de clasificación")

✅ Tabla silver.dim_cliente recreada con las 3 reglas de clasificación


## 4. Validación de la tabla generada

Se verifica que el resultado es coherente:

1. Que el número de filas es exactamente igual al original.
2. Que la clave primaria sigue siendo única.
3. Que los tipos de datos son los esperados.
4. Que las columnas nuevas tienen distribuciones razonables.

In [33]:
print("VALIDACIÓN DE INTEGRIDAD\n")

filas_bronze = con.execute("SELECT COUNT(*) FROM bronze.dim_cliente").fetchone()[0]
filas_silver = con.execute("SELECT COUNT(*) FROM silver.dim_cliente").fetchone()[0]
ids_unicos = con.execute("SELECT COUNT(DISTINCT id_cliente) FROM silver.dim_cliente").fetchone()[0]

print(f"Filas en bronze: {filas_bronze:,}")
print(f"Filas en silver: {filas_silver:,}")
print(f"id_cliente únicos: {ids_unicos:,}")
print(f"\nIntegridad de filas:  {'OK' if filas_bronze == filas_silver else 'ERROR'}")
print(f"Clave única:          {'OK' if filas_silver == ids_unicos else 'ERROR'}")

VALIDACIÓN DE INTEGRIDAD

Filas en bronze: 3,469
Filas en silver: 3,469
id_cliente únicos: 3,469

Integridad de filas:  OK
Clave única:          OK


In [34]:
print("ESQUEMA DE silver.dim_cliente\n")

esquema = con.execute("""
    SELECT column_name AS columna, data_type AS tipo
    FROM information_schema.columns
    WHERE table_schema = 'silver' AND table_name = 'dim_cliente'
    ORDER BY ordinal_position
""").fetchdf()

print(esquema.to_string(index=False))

ESQUEMA DE silver.dim_cliente

                 columna    tipo
              id_cliente VARCHAR
          nombre_cliente VARCHAR
nombre_comercial_cliente VARCHAR
       direccion_cliente VARCHAR
       localidad_cliente VARCHAR
  codigo_postal_original VARCHAR
      codigo_postal_norm VARCHAR
codigo_provincia_cliente VARCHAR
      es_cliente_espanol BOOLEAN
            tipo_mercado VARCHAR


In [35]:
print("DISTRIBUCIÓN DEL FLAG es_cliente_espanol\n")

dist_flag = con.execute("""
    SELECT 
        es_cliente_espanol,
        COUNT(*) AS num_clientes,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje
    FROM silver.dim_cliente
    GROUP BY es_cliente_espanol
    ORDER BY es_cliente_espanol DESC
""").fetchdf()

print(dist_flag.to_string(index=False))

DISTRIBUCIÓN DEL FLAG es_cliente_espanol

 es_cliente_espanol  num_clientes  porcentaje
               True          2082       60.02
              False          1387       39.98


In [36]:
print("DISTRIBUCIÓN DE tipo_mercado\n")

dist_mercado = con.execute("""
    SELECT 
        tipo_mercado,
        COUNT(*) AS num_clientes,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje
    FROM silver.dim_cliente
    GROUP BY tipo_mercado
    ORDER BY num_clientes DESC
""").fetchdf()

print(dist_mercado.to_string(index=False))

DISTRIBUCIÓN DE tipo_mercado

 tipo_mercado  num_clientes  porcentaje
     NACIONAL          2082       60.02
INTERNACIONAL          1387       39.98


In [37]:
print("VERIFICACIÓN DE LA NORMALIZACIÓN DEL CP\n")

verificacion_cp = con.execute("""
    SELECT 
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE codigo_postal_norm IS NULL) AS cp_norm_nulos,
        COUNT(*) FILTER (WHERE LENGTH(codigo_postal_norm) = 5 
                         AND REGEXP_MATCHES(codigo_postal_norm, '^[0-9]+$')) AS cp_norm_5_digitos,
        COUNT(*) FILTER (WHERE codigo_postal_norm IS NOT NULL 
                         AND (LENGTH(codigo_postal_norm) != 5 
                              OR NOT REGEXP_MATCHES(codigo_postal_norm, '^[0-9]+$'))) AS cp_norm_otros
    FROM silver.dim_cliente
""").fetchdf()

print(verificacion_cp.to_string(index=False))

VERIFICACIÓN DE LA NORMALIZACIÓN DEL CP

 total  cp_norm_nulos  cp_norm_5_digitos  cp_norm_otros
  3469              1               2910            558


In [38]:
print("EJEMPLOS DE CADA TIPO DE CLIENTE\n")

print("Clientes NACIONALES (5 ejemplos):")
ejemplos_es = con.execute("""
    SELECT id_cliente, nombre_cliente, localidad_cliente, 
           codigo_postal_original, codigo_postal_norm, codigo_provincia_cliente
    FROM silver.dim_cliente
    WHERE tipo_mercado = 'NACIONAL'
    LIMIT 5
""").fetchdf()
print(ejemplos_es.to_string(index=False))

print("\n\nClientes INTERNACIONALES (5 ejemplos):")
ejemplos_no_es = con.execute("""
    SELECT id_cliente, nombre_cliente, localidad_cliente,
           codigo_postal_original, codigo_postal_norm, codigo_provincia_cliente
    FROM silver.dim_cliente
    WHERE tipo_mercado = 'INTERNACIONAL'
    LIMIT 5
""").fetchdf()
print(ejemplos_no_es.to_string(index=False))

EJEMPLOS DE CADA TIPO DE CLIENTE

Clientes NACIONALES (5 ejemplos):
id_cliente           nombre_cliente localidad_cliente codigo_postal_original codigo_postal_norm codigo_provincia_cliente
         1  PEREZ RODRIGUEZ, AMADOR              VIGO                  36203              36203                      036
      1307        AROHA IBERICA, SL              VIGO                  36207              36207                       36
      1324 PATRICIA GONZALEZ FERRAL              VIGO                  36214              36214                       36
        17    PEREZ GONZALEZ, JORGE              Vigo                  36305              36305                       36
      1964          SELMARK, S.L.U.              VIGO                  36315              36315                      036


Clientes INTERNACIONALES (5 ejemplos):
id_cliente             nombre_cliente localidad_cliente codigo_postal_original codigo_postal_norm codigo_provincia_cliente
      1010       PERFECT FIT LINGERIE     

In [39]:
# Investigar cuántos clientes tienen CP español pero provincia con formato no estándar
print("ANÁLISIS DE CASOS LÍMITE: CP español + provincia con formato distinto\n")

casos_limite = con.execute("""
    WITH analisis AS (
        SELECT 
            id_cliente,
            nombre_cliente,
            localidad_cliente,
            codigo_postal_original,
            codigo_postal_norm,
            codigo_provincia_cliente,
            tipo_mercado,
            -- Extraer los 2 primeros dígitos del CP normalizado (si es numérico)
            CASE 
                WHEN LENGTH(codigo_postal_norm) = 5 
                     AND REGEXP_MATCHES(codigo_postal_norm, '^[0-9]{5}$')
                THEN SUBSTR(codigo_postal_norm, 1, 2)
                ELSE NULL
            END AS provincia_segun_cp
        FROM silver.dim_cliente
    )
    SELECT 
        provincia_segun_cp,
        codigo_provincia_cliente,
        tipo_mercado,
        COUNT(*) AS num_clientes
    FROM analisis
    WHERE provincia_segun_cp IS NOT NULL
      AND TRY_CAST(provincia_segun_cp AS INTEGER) BETWEEN 1 AND 52
      AND tipo_mercado = 'INTERNACIONAL'
    GROUP BY provincia_segun_cp, codigo_provincia_cliente, tipo_mercado
    ORDER BY num_clientes DESC
    LIMIT 20
""").fetchdf()

print("Top 20 casos límite (clientes con CP español pero marcados como internacionales):\n")
print(casos_limite.to_string(index=False))

# Total de casos
total_casos = con.execute("""
    SELECT COUNT(*) AS total
    FROM silver.dim_cliente
    WHERE LENGTH(codigo_postal_norm) = 5 
      AND REGEXP_MATCHES(codigo_postal_norm, '^[0-9]{5}$')
      AND TRY_CAST(SUBSTR(codigo_postal_norm, 1, 2) AS INTEGER) BETWEEN 1 AND 52
      AND tipo_mercado = 'INTERNACIONAL'
""").fetchone()[0]

print(f"\nTotal de casos límite: {total_casos} clientes")

ANÁLISIS DE CASOS LÍMITE: CP español + provincia con formato distinto

Top 20 casos límite (clientes con CP español pero marcados como internacionales):

provincia_segun_cp codigo_provincia_cliente  tipo_mercado  num_clientes
                20                      410 INTERNACIONAL            24
                16                      410 INTERNACIONAL            23
                25                      410 INTERNACIONAL            17
                02                      471 INTERNACIONAL            15
                01                       NO INTERNACIONAL            15
                09                      471 INTERNACIONAL            13
                40                      410 INTERNACIONAL            12
                08                      471 INTERNACIONAL            12
                03                      471 INTERNACIONAL            11
                03                       NO INTERNACIONAL            10
                21                      410 INTERNACIO

In [40]:
# Investigar las LOCALIDADES de los casos límite para entender qué son realmente
print("LOCALIDADES DE LOS CASOS LÍMITE\n")

print("=== CASO 1: provincia_segun_cp=36 + provincia_cliente='036' (los del tipo PEREZ RODRIGUEZ) ===")
caso_036 = con.execute("""
    SELECT id_cliente, nombre_cliente, localidad_cliente, codigo_postal_norm, codigo_provincia_cliente
    FROM silver.dim_cliente
    WHERE SUBSTR(codigo_postal_norm, 1, 2) = '36'
      AND codigo_provincia_cliente = '036'
      AND tipo_mercado = 'INTERNACIONAL'
    LIMIT 10
""").fetchdf()
print(caso_036.to_string(index=False))

print("\n\n=== CASO 2: CP español + provincia='410' (sospechosos de extranjeros) ===")
caso_410 = con.execute("""
    SELECT id_cliente, nombre_cliente, localidad_cliente, codigo_postal_norm, codigo_provincia_cliente
    FROM silver.dim_cliente
    WHERE codigo_provincia_cliente = '410'
      AND tipo_mercado = 'INTERNACIONAL'
    LIMIT 10
""").fetchdf()
print(caso_410.to_string(index=False))

print("\n\n=== CASO 3: CP español + provincia='NO' (Noruega?) ===")
caso_no = con.execute("""
    SELECT id_cliente, nombre_cliente, localidad_cliente, codigo_postal_norm, codigo_provincia_cliente
    FROM silver.dim_cliente
    WHERE codigo_provincia_cliente = 'NO'
      AND tipo_mercado = 'INTERNACIONAL'
    LIMIT 10
""").fetchdf()
print(caso_no.to_string(index=False))

print("\n\n=== CASO 4: CP español + provincia=NULL ===")
caso_null = con.execute("""
    SELECT id_cliente, nombre_cliente, localidad_cliente, codigo_postal_norm, codigo_provincia_cliente
    FROM silver.dim_cliente
    WHERE codigo_provincia_cliente IS NULL
      AND LENGTH(codigo_postal_norm) = 5
      AND REGEXP_MATCHES(codigo_postal_norm, '^[0-9]{5}$')
      AND TRY_CAST(SUBSTR(codigo_postal_norm, 1, 2) AS INTEGER) BETWEEN 1 AND 52
    LIMIT 10
""").fetchdf()
print(caso_null.to_string(index=False))

LOCALIDADES DE LOS CASOS LÍMITE

=== CASO 1: provincia_segun_cp=36 + provincia_cliente='036' (los del tipo PEREZ RODRIGUEZ) ===
Empty DataFrame
Columns: [id_cliente, nombre_cliente, localidad_cliente, codigo_postal_norm, codigo_provincia_cliente]
Index: []


=== CASO 2: CP español + provincia='410' (sospechosos de extranjeros) ===
id_cliente                    nombre_cliente             localidad_cliente codigo_postal_norm codigo_provincia_cliente
      1470             SORTINO GIAN BATTISTA CANALICCHIO TREMESTIERI ETNEO              95030                      410
      1471                ALESSANDRO CARRARO      CARDANO AL CAMPO  VARESE              21010                      410
      1472   VANITY SNC DI OLLA GINETTA & C.                        NOVARA              28100                      410
      1474 IL BACO DA SETA DI GIUNTA DANIELA                        PESARO              61122                      410
      1476                AROSIO MARIAGRAZIA                      BIASSO

In [41]:
# Calcular cuántos clientes serían rescatados con la nueva regla
print("CLIENTES QUE SERÍAN RESCATADOS COMO NACIONALES\n")

rescatados = con.execute("""
    SELECT 
        codigo_provincia_cliente AS prov_actual,
        SUBSTR(codigo_postal_norm, 1, 2) AS prov_segun_cp,
        COUNT(*) AS num_clientes,
        COUNT(DISTINCT localidad_cliente) AS localidades_distintas
    FROM silver.dim_cliente
    WHERE LENGTH(codigo_postal_norm) = 5
      AND REGEXP_MATCHES(codigo_postal_norm, '^[0-9]{5}$')
      AND TRY_CAST(SUBSTR(codigo_postal_norm, 1, 2) AS INTEGER) BETWEEN 1 AND 52
      -- Provincia de 3 dígitos numéricos que empieza por 0
      AND LENGTH(codigo_provincia_cliente) = 3
      AND REGEXP_MATCHES(codigo_provincia_cliente, '^0[0-9]{2}$')
      -- Y los 2 últimos dígitos coinciden con los 2 primeros del CP
      AND SUBSTR(codigo_provincia_cliente, 2, 2) = SUBSTR(codigo_postal_norm, 1, 2)
      AND tipo_mercado = 'INTERNACIONAL'
    GROUP BY codigo_provincia_cliente, SUBSTR(codigo_postal_norm, 1, 2)
    ORDER BY num_clientes DESC
""").fetchdf()

print(rescatados.to_string(index=False))
print(f"\nTotal clientes rescatados: {rescatados['num_clientes'].sum()}")

CLIENTES QUE SERÍAN RESCATADOS COMO NACIONALES

Empty DataFrame
Columns: [prov_actual, prov_segun_cp, num_clientes, localidades_distintas]
Index: []

Total clientes rescatados: 0


## 5. Análisis geográfico tras la limpieza

Se verifica la distribución geográfica de los clientes nacionales por código de provincia. Esta información será útil para el análisis de geomarketing.

In [42]:
print("TOP 15 PROVINCIAS POR NÚMERO DE CLIENTES (clientes nacionales)\n")

top_provincias = con.execute("""
    SELECT 
        codigo_provincia_cliente AS cod_provincia,
        COUNT(*) AS num_clientes
    FROM silver.dim_cliente
    WHERE tipo_mercado = 'NACIONAL'
    GROUP BY codigo_provincia_cliente
    ORDER BY num_clientes DESC
    LIMIT 15
""").fetchdf()

print(top_provincias.to_string(index=False))

TOP 15 PROVINCIAS POR NÚMERO DE CLIENTES (clientes nacionales)

cod_provincia  num_clientes
           08           226
          NaN           213
           36            95
           33            93
           15            83
           28            83
           46            80
           38            56
           03            54
           43            53
           41            52
           17            49
           20            49
           48            47
           50            43


## 5.1 Análisis exhaustivo del campo `codigo_provincia_cliente`

Tras analizar en detalle los códigos del campo `codigo_provincia_cliente` en bronze, se han identificado los siguientes formatos:

| Formato | Clientes | Interpretación |
|---|---|---|
| 2 dígitos numéricos (01-52) | 1.854 | Provincias españolas estándar |
| 3 dígitos numéricos | 1.169 | Códigos numéricos de país en codificación interna de Selmark |
| 2 letras | 151 | Códigos ISO de país (PT, FR, IT, etc.) |
| 3 letras | 57 | Códigos ISO de país (CAN, USA, etc.) |
| Otro formato / nulos | 238 | Sin clasificar |

### Investigación de los códigos numéricos de 3 dígitos

Se ha realizado una investigación específica para entender el significado de los códigos de 3 dígitos numéricos. La investigación cruzando código de provincia con la localidad real del cliente ha permitido identificar lo siguiente:

| Código | Casos | Significado real |
|---|---|---|
| `410` | 441 | **Italia** (TREMESTIERI ETNEO, NOVARA, PESARO, MESSINA, etc.) |
| `231` | 256 | País extranjero (a confirmar con tutor) |
| `000` | 83 | Polonia mayoritariamente (KRAKOW, GDYNIA, TORUN, etc.) |
| `471` | 60 | País extranjero (a confirmar con tutor) |
| `036` | 22 | **España (Pontevedra)**: error de codificación; debería ser '36' |
| `028` | 2 | **España (Madrid)**: error de codificación; debería ser '28' |

### Caso límite identificado: códigos '0XX' equivalentes a provincias españolas

Durante la investigación se ha detectado un grupo reducido de clientes con código de provincia tipo `0XX` (donde XX es un número de provincia española) cuyo CP es coherente con esa provincia y cuya localidad es claramente española. Tras analizar caso por caso, se confirma que son **clientes españoles con un error de carga en el ERP**, donde el código de provincia se almacenó como tres dígitos en lugar de dos.

Algunos ejemplos representativos:

- `id_cliente=1`: PEREZ RODRIGUEZ, AMADOR — VIGO — CP 36203 — provincia '036'
- `id_cliente=1964`: SELMARK, S.L.U. — VIGO — CP 36315 — provincia '036'
- 22 clientes en total con patrón `036` (Pontevedra)
- 2 clientes en total con patrón `028` (Madrid)

### Lógica final del flag `es_cliente_espanol`

Para no perder estos clientes españoles legítimos, la lógica del flag se ha ampliado con dos reglas combinadas mediante operador OR:

**Regla A (formato estándar)**: el código de provincia es de 2 dígitos numéricos en el rango 01-52 y coincide con los dos primeros dígitos del CP normalizado.

**Regla B (formato alternativo '0XX')**: el código de provincia es de 3 dígitos del patrón `0XX` y los dos últimos dígitos coinciden con los dos primeros del CP normalizado.

Ambas reglas exigen además que el CP sea numérico de hasta 5 dígitos. Esta validación cruzada es **necesaria** porque países como Italia, Noruega o Polonia tienen códigos postales numéricos de cinco dígitos que, tras el padding, podrían confundirse con CPs españoles si no se contrastaran con un segundo dato (la provincia).

Tras aplicar esta lógica refinada, se han recuperado **24 clientes españoles** que estaban incorrectamente marcados como internacionales, incluyendo a la propia matriz `SELMARK, S.L.U.`

## 5.2 Decisión sobre la tipificación de clientes (a desarrollar en Gold)

Tras la conversación con el tutor, se confirma que la diferenciación nacional/internacional **no es suficiente** para abordar correctamente los análisis posteriores de clustering y machine learning. Selmark opera con una cartera que combina canales muy heterogéneos:

| Canal | Naturaleza | Comportamiento esperado |
|---|---|---|
| B2B nacional | Tiendas multimarca, distribuidores | Pedidos voluminosos, regularidad, ticket alto |
| El Corte Inglés (ECI) | Operativa de venta a gran cuenta | Pedidos masivos, devoluciones gestionadas |
| Ecommerce | Compras de particulares vía web | Tickets pequeños, alta frecuencia, sin previsibilidad |
| Depósito | Operativa logística específica | Movimientos de stock, no venta directa |
| Muestras | Comerciales y promocionales | Volumen bajo, sin facturación directa |
| Distribución internacional | Partners y exportación | Volúmenes muy variables, calendarios distintos |

### Implicación para los modelos

Mezclar todos los canales en un mismo clustering produciría grupos dominados por las diferencias de canal en lugar de por el comportamiento real del cliente dentro de su canal. Por ejemplo, un buen cliente de Ecommerce y un buen cliente B2B son comparables solo dentro de su segmento, no entre segmentos.

### Estrategia de implementación

La tipología de cliente (`tipo_cliente`) **no se construirá en Silver**, ya que requiere agregar las operaciones de cada cliente desde las tablas de hechos. Se calculará en la capa Gold (`gold.cliente_360`) a partir del mix de tipos de operación realizadas, asignando a cada cliente su perfil dominante. Posteriormente, los análisis de clustering y machine learning se ejecutarán dentro de cada tipo y no sobre el conjunto completo de la cartera.

Esta decisión metodológica garantiza que los modelos posteriores capturen patrones reales de comportamiento dentro de poblaciones homogéneas, en lugar de diferencias estructurales entre canales.

## 6. Conclusiones y siguientes pasos

### Resultado obtenido

Se ha generado la tabla `silver.dim_cliente` con 3.469 registros (mismo volumen que la tabla origen), enriquecida con los siguientes aportes:

- Normalización del código postal a cinco dígitos para los clientes españoles, manteniendo el formato original como columna paralela para trazabilidad.
- Flag booleano `es_cliente_espanol` y categoría textual `tipo_mercado` (`NACIONAL` / `INTERNACIONAL`) que permiten segmentar la cartera en sus dos grandes bloques.
- Limpieza de espacios en blanco en todos los campos de texto.

### Composición de la cartera

| Tipo de mercado | Clientes | Porcentaje |
|---|---|---|
| NACIONAL | 1.845 | 53,2 % |
| INTERNACIONAL | 1.624 | 46,8 % |

### Estrategia de uso de la cartera en el TFG

La cartera de Selmark presenta dos dimensiones de heterogeneidad que deben tratarse explícitamente:

**Dimensión geográfica (`tipo_mercado`)**:
- NACIONAL (1.845 clientes): población objetivo del análisis técnico (clustering, geomarketing, ML).
- INTERNACIONAL (1.624 clientes): mayoritariamente vinculados al canal ecommerce y a la exportación; se incluirán solo en la sección de análisis estratégico de negocio.

**Dimensión de canal (`tipo_cliente`)**:
- Se construirá en la capa Gold a partir del mix de operaciones de cada cliente.
- Permitirá segmentar la cartera nacional en perfiles homogéneos (B2B, ECI, Ecommerce nacional, Depósito, etc.) sobre los que aplicar clustering y ML de forma metodológicamente correcta.

Los análisis posteriores combinarán ambas dimensiones para garantizar que los modelos se entrenen sobre poblaciones internamente homogéneas.

### Validaciones superadas

- Volumen idéntico al origen (sin pérdida ni duplicación de filas).
- Clave primaria `id_cliente` única.
- Tipos de datos coherentes (VARCHAR en identificadores, BOOLEAN en flags).
- El flag distingue correctamente entre provincias españolas estándar y códigos numéricos de país.

### Implicaciones para los siguientes notebooks

- Esta tabla será la dimensión de referencia para todos los JOINs con `id_cliente` en notebooks posteriores.
- El campo `codigo_postal_norm` será la clave de unión con `silver.mosaic`.
- El filtro `tipo_mercado = 'NACIONAL'` se aplicará por defecto en los análisis técnicos.

### Próximo notebook

`04_silver_fact_lineas_pedido.ipynb` — Construcción de la tabla de hechos de líneas de pedido limpia, aplicando la ventana temporal 2022-2025, excluyendo líneas anuladas y calculando los días entre pedido y entrega.

In [43]:
# ============================================================
# CIERRE DE LA SESIÓN
# ============================================================
# Libera la conexión a DuckDB para evitar bloqueos en otros notebooks

try:
    con.close()
    print("Conexión a DuckDB cerrada correctamente.")
except Exception as e:
    print(f"Aviso al cerrar conexión: {e}")

import gc
gc.collect()

print("\nTabla silver.dim_cliente persistida en disco.")
print("La base está libre para otros notebooks.")
print("Recomendación: 'Kernel -> Shutdown' antes de abrir el siguiente notebook.")

Conexión a DuckDB cerrada correctamente.

Tabla silver.dim_cliente persistida en disco.
La base está libre para otros notebooks.
Recomendación: 'Kernel -> Shutdown' antes de abrir el siguiente notebook.
